# GemCol Evaluation: Phase 1 (Setup)
This notebook sets up the environment and recreates the utility files.

In [ ]:
import os
from pathlib import Path

folders = [
    '/workspace/gemcol_evaluation/utils',
    '/workspace/data/msmarco',
    '/workspace/data/beir',
    '/workspace/checkpoints',
    '/workspace/results',
    '/workspace/figures',
    '/workspace/notebooks',
    '/tmp/gemcol'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    
print('✅ Folder structure created successfully!')

In [ ]:
!pip install rank-bm25 tqdm ipywidgets pyserini qdrant-client sentence-transformers datasets
!pip install git+https://github.com/stanford-futuredata/ColBERT.git

In [ ]:
%%writefile /workspace/gemcol_evaluation/utils/config.py
from pathlib import Path

PATHS = {
    "base": Path("/workspace"),
    "utils": Path("/workspace/gemcol_evaluation/utils"),
    "data": Path("/workspace/data"),
    "msmarco": Path("/workspace/data/msmarco"),
    "beir": Path("/workspace/data/beir"),
    "checkpoints": Path("/workspace/checkpoints"),
    "results": Path("/workspace/results"),
    "figures": Path("/workspace/figures"),
    "notebooks": Path("/workspace/notebooks"),
    "tmp": Path("/tmp/gemcol")
}

MSMARCO = {
    "queries": PATHS["msmarco"] / "queries.dev.small.tsv",
    "qrels": PATHS["msmarco"] / "qrels.dev.small.tsv",
    "collection": PATHS["msmarco"] / "collection.tsv",
    "tar": PATHS["msmarco"] / "collection.tar.gz"
}

DEFAULT_CONFIG = {
    "bm25": {"k1": 0.9, "b": 0.4, "top_k": 100},
    "bge": {"model": "BAAI/bge-large-en-v1.5", "batch_size": 256, "top_k": 100},
    "rrf": {"k": 60, "top_k": 100},
    "colbert": {"model": "colbert-ir/colbertv2.0", "top_k": 10}
}

ABLATION = {"rrf_k_values": [10, 30, 60, 100], "pool_sizes": [25, 50, 100, 200]}

EXPECTED_NDCG = {
    "bm25": (0.28, 0.30),
    "bge": (0.35, 0.40),
    "fusion": (0.32, 0.35),
    "gemcol": (0.42, 0.45)
}

def sanity_check_ndcg(system, score):
    expected = EXPECTED_NDCG.get(system)
    if expected and not (expected[0] <= score <= expected[1]):
        print(f"WARNING: {system} nDCG@10 ({score:.4f}) is outside expected range {expected}!")

BEIR_DATASETS = ["nfcorpus", "scifact", "trec-covid", "arguana"]


In [ ]:
%%writefile /workspace/gemcol_evaluation/utils/metrics.py
import math
import json

def ndcg_at_k(ranked_doc_ids, qrels, k=10):
    dcg = 0.0
    for i, doc_id in enumerate(ranked_doc_ids[:k]):
        rel = qrels.get(doc_id, 0)
        dcg += (2**rel - 1) / math.log2(i + 2)
    idcg = (2**1 - 1) / math.log2(2)
    return dcg / idcg if idcg > 0 else 0.0

def mrr_at_k(ranked_doc_ids, qrels, k=10):
    for i, doc_id in enumerate(ranked_doc_ids[:k]):
        if qrels.get(doc_id, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0

def recall_at_k(ranked_doc_ids, qrels, k=100):
    for doc_id in ranked_doc_ids[:k]:
        if qrels.get(doc_id, 0) > 0:
            return 1.0
    return 0.0

def evaluate_run(run, qrels_all):
    metrics = {"ndcg@10": 0, "mrr@10": 0, "recall@100": 0}
    count = 0
    for qid, ranked_docs in run.items():
        if qid not in qrels_all: continue
        qrels = qrels_all[qid]
        metrics["ndcg@10"] += ndcg_at_k(ranked_docs, qrels, 10)
        metrics["mrr@10"] += mrr_at_k(ranked_docs, qrels, 10)
        metrics["recall@100"] += recall_at_k(ranked_docs, qrels, 100)
        count += 1
    if count > 0:
        metrics = {k: v / count for k, v in metrics.items()}
    return metrics

def print_metrics(metrics, system_name):
    print(f"--- {system_name} Metrics ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

def compare_systems(results_dict):
    pass

def save_metrics(metrics, path):
    with open(path, 'w') as f: json.dump(metrics, f)

def load_metrics(path):
    with open(path, 'r') as f: return json.load(f)


In [ ]:
%%writefile /workspace/gemcol_evaluation/utils/data_loader.py
import json
from .config import MSMARCO, PATHS
from tqdm import tqdm
from datasets import load_dataset
import urllib.request

def load_msmarco_dev(data_dir=None):
    print("Loading queries via HF Hub (Tevatron/msmarco-passage)...")
    dev_ds = load_dataset("Tevatron/msmarco-passage", split="validation")
    queries = {str(row["query_id"]): str(row["query"]) for row in dev_ds}
    
    print("Downloading qrels from BeIR (to fix missing Azure blob)...")
    req = urllib.request.urlopen("https://huggingface.co/datasets/BeIR/msmarco-qrels/resolve/main/dev.tsv")
    qrels = {}
    for line in req.read().decode("utf-8").splitlines()[1:]:
        qid, docid, score = line.split("\t")
        if qid not in qrels: qrels[qid] = {}
        qrels[qid][docid] = int(score)
            
    print("Loading 8.8M corpus via HF Hub (Tevatron/msmarco-passage-corpus)...")
    corpus_ds = load_dataset("Tevatron/msmarco-passage-corpus", split="train")
    
    corpus = {}
    for row in tqdm(corpus_ds, desc="Building corpus dictionary"):
        corpus[str(row["docid"])] = str(row["text"])
        
    return queries, qrels, corpus

def load_beir_dataset(name, beir_dir):
    pass

def beir_corpus_to_texts(corpus):
    return {docid: doc.get("title", "") + " " + doc.get("text", "") for docid, doc in corpus.items()}

def save_run_json(run, path):
    with open(path, 'w') as f: json.dump(run, f)

def load_run_json(path):
    with open(path, 'r') as f: return json.load(f)

def save_run(run, path): save_run_json(run, path)
def load_run(path): return load_run_json(path)


In [ ]:
%%writefile /workspace/gemcol_evaluation/utils/retrieval.py
import json
import time
import os
import pickle
from collections import defaultdict
from .config import PATHS

def rrf_fusion(runs, k=60, top_k=100):
    fused_scores = defaultdict(lambda: defaultdict(float))
    for run in runs:
        for qid, docs in run.items():
            for rank, docid in enumerate(docs):
                fused_scores[qid][docid] += 1.0 / (k + rank + 1)
    
    fused_run = {}
    for qid, doc_scores in fused_scores.items():
        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        fused_run[qid] = [docid for docid, score in sorted_docs[:top_k]]
    return fused_run

def rrf_with_scores(runs, k=60, top_k=100):
    pass

def retrieval_source_breakdown(bm25_run, bge_run, qrels):
    pass 

def save_checkpoint(obj, name):
    path = PATHS["checkpoints"] / f"{name}.pkl"
    with open(path, 'wb') as f: pickle.dump(obj, f)

def load_checkpoint(name):
    path = PATHS["checkpoints"] / f"{name}.pkl"
    with open(path, 'rb') as f: return pickle.load(f)

def checkpoint_exists(name):
    return (PATHS["checkpoints"] / f"{name}.pkl").exists()

def load_or_compute(name, fn):
    if checkpoint_exists(name):
        print(f"Loading checkpoint {name}...")
        return load_checkpoint(name)
    print(f"Computing {name}...")
    obj = fn()
    save_checkpoint(obj, name)
    return obj

class LatencyProfiler:
    def __init__(self):
        self.times = defaultdict(list)
    class _Timer:
        def __init__(self, profiler, name):
            self.profiler = profiler
            self.name = name
        def __enter__(self):
            self.start = time.time()
        def __exit__(self, exc_type, exc_val, exc_tb):
            self.profiler.times[self.name].append((time.time() - self.start)*1000)
    def time(self, name):
        return self._Timer(self, name)
    def summary(self):
        import numpy as np
        res = {}
        for k, v in self.times.items():
            res[k] = {"mean_ms": np.mean(v), "p95_ms": np.percentile(v, 95), "p99_ms": np.percentile(v, 99)}
        return res
    def print_summary(self):
        for k, v in self.summary().items():
            print(f"{k}: Mean={v['mean_ms']:.2f}ms, P95={v['p95_ms']:.2f}ms, P99={v['p99_ms']:.2f}ms")

class ExperimentTracker:
    def __init__(self, path):
        self.path = path
        self.results = self._load()
    def _load(self):
        if os.path.exists(self.path):
            with open(self.path, 'r') as f: return json.load(f)
        return {}
    def log(self, name, config, metrics):
        self.results[name] = {"config": config, "metrics": metrics}
        with open(self.path, 'w') as f: json.dump(self.results, f, indent=2)


In [ ]:
%%writefile /workspace/gemcol_evaluation/utils/__init__.py
from .config import *
from .metrics import *
from .data_loader import *
from .retrieval import *

print('✅ All utility files written to /workspace/gemcol_evaluation/utils')